<a href="https://colab.research.google.com/github/LightBolt8/politicianEmotions/blob/main/preprocessing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
!pip install face_recognition --quiet
import face_recognition

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!wget -O candidate_A.jpg "https://upload.wikimedia.org/wikipedia/commons/5/5d/Donald_Trump_close-up_%28cropped%29.jpg" --quiet

In [ ]:
!wget -O candidate_B.jpg "https://tile.loc.gov/storage-services/service/pnp/ppbd/01200/01262r.jpg" --quiet

In [ ]:
#have to use load_image_file to read image file first because face_encodings function can't open files on its own
img_A = face_recognition.load_image_file("/content/candidate_A.jpg")
img_B = face_recognition.load_image_file("/content/candidate_B.jpg")

In [ ]:
#image encoding is unique digital fingerprint of human face turned into list of numbers
#raw photo to 68 facial landmarks to align face so its looking forward
#sends aligned face through ResNet CNN to measure distance between features: 128 numbers
temp_encoding_A = face_recognition.face_encodings(img_A)
temp_encoding_A

[array([-0.14468589,  0.12666398,  0.01114747,  0.00755378, -0.11907183,
        -0.03951233,  0.04275287, -0.17542976,  0.07950877, -0.09651626,
         0.19032979, -0.08675314, -0.36444163, -0.0755813 ,  0.0082342 ,
         0.137318  , -0.14856397, -0.16525616, -0.22993743, -0.13619955,
         0.02846617, -0.00050909, -0.02842057, -0.03471612, -0.12013651,
        -0.23386906, -0.05190622, -0.14582829,  0.01764276, -0.08940136,
         0.05386482, -0.01859802, -0.19446182, -0.12630457,  0.00324349,
        -0.01856636, -0.08913778, -0.06496169,  0.18091138, -0.02954188,
        -0.16361029, -0.01685644,  0.02441896,  0.21379848,  0.21936795,
         0.02647728,  0.00813737, -0.16995744,  0.09407459, -0.28874651,
        -0.00558806,  0.14656022,  0.14559336,  0.11680931,  0.07729152,
        -0.14256747,  0.04802712,  0.12924917, -0.22693582,  0.06501116,
         0.0818344 , -0.14639363, -0.04672801, -0.04924441,  0.12000562,
         0.08336293, -0.02516322, -0.11976949,  0.2

In [ ]:
temp_encoding_B = face_recognition.face_encodings(img_B)
temp_encoding_B

[array([-0.10640474,  0.10078781,  0.1436509 , -0.06866901, -0.06462405,
        -0.06616384, -0.06603629, -0.16493611,  0.10867272,  0.01982388,
         0.2477496 , -0.03125939, -0.2456335 , -0.02448015, -0.00673858,
         0.14700198, -0.10915336, -0.15918134, -0.14944109,  0.00655629,
        -0.03794369,  0.00425723,  0.08782762,  0.12871188, -0.09875563,
        -0.39821509, -0.05839239, -0.13568905,  0.05598528,  0.01996426,
        -0.02424087,  0.04157616, -0.23628885, -0.03926209,  0.10060663,
         0.12523796,  0.00082492, -0.03183926,  0.16582321, -0.05119586,
        -0.26847863,  0.0542727 ,  0.09943002,  0.17984949,  0.21424133,
         0.01322385,  0.13211711, -0.13585964,  0.04785634, -0.19848137,
         0.13493791,  0.13296752,  0.07023348,  0.05452568,  0.11477599,
        -0.23781435, -0.03891236,  0.17565972, -0.14841136,  0.1074791 ,
         0.04535931, -0.02538352, -0.04514032, -0.06872986,  0.2211042 ,
         0.12230927, -0.14485458, -0.14536315,  0.2

In [ ]:
#turn first item in array of 1 item into its own list
encoding_A = temp_encoding_A[0]
encoding_B = temp_encoding_B[0]
known_encodings = [encoding_A, encoding_B]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#unzip video file into frame by frame pictures
input_video = cv2.VideoCapture("/content/drive/MyDrive/Trump vs Harris")
print(input_video)

< cv2.VideoCapture 0x7b5c7d34d750>


In [ ]:
fps=5
fourcc=cv2.VideoWriter_fourcc(*"mp4v")
frameSize=(256,256)

In [ ]:
#creates candidate_X_clean that only contains certain candidate A's face cropped
output_A=cv2.VideoWriter("candidate_A_clean.mp4", fourcc, fps, frameSize)
output_B=cv2.VideoWriter("candidate_B_clean.mp4", fourcc, fps, frameSize)
print(output_A)
print(output_B)

< cv2.VideoWriter 0x7b5c7d34e330>
< cv2.VideoWriter 0x7b5c7d34fe70>


In [ ]:
total_frames = int(input_video.get(cv2.CAP_PROP_FRAME_COUNT))
skip_rate=6
print(total_frames)
#grab every 6th frame in a 30fps video: 5fps

0


In [ ]:
frame_count=0
for _ in range (total_frames):
  ret, frame = input_video.read()
  print(ret, frame)
  if not ret:
    break #end video if no more frames left
  frame_count+=1
  if frame_count % 600 ==0: #print progress every 500 frames
    print(f"Currently processing {frame_count} / {total_frames}")
  if frame_count % skip_rate !=0: #skip 5 frames every 6
    continue
  #convert color to RGB
  rgb_frame=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  #finds faces and creates boxes around them
  face_locations=face_recognition.face_locations(rgb_frame)
  #only scans in boxes and finds 128 number encoding for each face
  face_encodings=face_recognition.face_encodings(rgb_frame, face_locations)

  #for (coordinates of face bounding box), loop through 2 separate lists
  #checks if first index in face_locations belongs to candidate 1 (first loop), candidate 2 (second loop)
  for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(known_encodings, face_encoding, tolerance=0.5)
    height, width, _=frame.shape
    pad = int((bottom-top)*0.4) #adds 40% safety cushion so don't cut off part of face
    #uses pad to expand cropping box by adding pad value to all
    y1,y2=max(0,top-pad),min(height,bottom+pad)
    x1,x2=max(0,left-pad),min(width,right+pad)
    cropped_face=frame[y1:y2,x1:x2] #cuts out square section in newly padded coordinates
    if cropped_face.size==0: #if cropped face is 0 pixels
      continue
    resized_face=cv2.resize(cropped_face, frameSize) #scale isolated face box to frameSize

    if matches[0]: #index 0 is candidate A, face pic into candidate_A_clean
      output_A.write(resized_face)
    elif matches[1]: #index 1 is candidate B, face pic into candidate_B_clean
      output_B.write(resized_face)


In [ ]:
#close files
input_video.release()
output_A.release()
output_B.release()